In [ ]:

import glob, json, os, subprocess, sys, time, urllib.request

MAX_GB = 88.0          # больше не влезет в 96 ГБ вместе с KV-кэшем
CTX = 4096
PARALLEL = 2

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw).stdout

print("[[DSV4]] GPU:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip(), flush=True)
print("[[DSV4]] диск:", sh("df -h /kaggle/working | tail -1").strip(), flush=True)
print("[[DSV4]] входы:", sorted(os.listdir("/kaggle/input")), flush=True)

# --- 1. что реально лежит во входах ---
gguf = []
for root, _dirs, files in os.walk("/kaggle/input"):
    for f in files:
        p = os.path.join(root, f)
        try:
            sz = os.path.getsize(p) / 1e9
        except OSError:
            continue
        if f.endswith(".gguf"):
            gguf.append((sz, p))
gguf.sort(reverse=True)
print("[[DSV4]] найдено файлов .gguf: %d" % len(gguf), flush=True)
for sz, p in gguf[:25]:
    print("   %7.1f ГБ  %s" % (sz, p), flush=True)

server_bin = None
for root, _dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "llama-server":
            server_bin = os.path.join(root, f)
print("[[DSV4]] llama-server:", server_bin, flush=True)

fit = [(sz, p) for sz, p in gguf if sz <= MAX_GB]
if not fit or server_bin is None:
    print("[[DSV4]] ИТОГ: нечего запускать (подходящий квант или сервер не найдены)", flush=True)
else:
    sz, model = fit[0]          # самый крупный из влезающих
    print("[[DSV4]] выбран квант: %.1f ГБ %s" % (sz, model), flush=True)
    os.chmod(server_bin, 0o755)
    for lib in glob.glob(os.path.dirname(server_bin) + "/../lib*"):
        os.environ["LD_LIBRARY_PATH"] = lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    cmd = ("%s -m %s --host 127.0.0.1 --port 8080 -ngl 999 -c %d --parallel %d"
           "--flash-attn on --log-disable" % (server_bin, model, CTX, PARALLEL))
    print("[[DSV4]] запуск:", cmd, flush=True)
    t0 = time.time()
    log = open("/kaggle/working/llama-server.log", "w")
    proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)
    ok = False
    while time.time() - t0 < 1800:
        time.sleep(10)
        if proc.poll() is not None:
            print("[[DSV4]] сервер упал через %.0f с, код %s" % (time.time() - t0, proc.returncode), flush=True)
            break
        try:
            with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5) as r:
                if r.status == 200:
                    ok = True; break
        except Exception:
            pass
        if int(time.time() - t0) % 120 < 10:
            print("[[DSV4]] жду сервер %.0f с; %s" % (time.time() - t0, sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader").strip()), flush=True)
    load_s = time.time() - t0
    print("[[DSV4]] сервер поднялся: %s, загрузка %.0f с" % (ok, load_s), flush=True)
    if ok:
        body = json.dumps({"model": "dsv4", "messages": [
            {"role": "user", "content": "You see a 3x3 grid of coloured cells. Reply with one short sentence: what "
                                        "information would you need to infer the goal of an unknown puzzle game?"}],
            "max_tokens": 200, "temperature": 0.6}).encode()
        t1 = time.time(); out = None
        try:
            req = urllib.request.Request("http://127.0.0.1:8080/v1/chat/completions", data=body,
                                         headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=600) as r:
                out = json.loads(r.read())
        except Exception as exc:
            print("[[DSV4]] запрос не прошёл: %r" % (exc,), flush=True)
        dt = time.time() - t1
        if out:
            txt = out["choices"][0]["message"]["content"]
            n = out.get("usage", {}).get("completion_tokens", 0)
            print("[[DSV4]] ответ за %.1f с, токенов %s, скорость %.1f ток/с" % (dt, n, n / max(dt, 1e-9)), flush=True)
            print("[[DSV4]] текст ответа: %s" % txt[:400].replace("\n", " "), flush=True)
        # параллельная нагрузка: 4 запроса разом
        import threading
        res = []
        def one(i):
            b = json.dumps({"model": "dsv4", "messages": [{"role": "user", "content": "Count from 1 to 40, comma separated."}],
                            "max_tokens": 200, "temperature": 0.6}).encode()
            try:
                rq = urllib.request.Request("http://127.0.0.1:8080/v1/chat/completions", data=b,
                                            headers={"Content-Type": "application/json"})
                with urllib.request.urlopen(rq, timeout=900) as r:
                    res.append(json.loads(r.read()).get("usage", {}).get("completion_tokens", 0))
            except Exception as exc:
                res.append(0)
        t2 = time.time()
        th = [threading.Thread(target=one, args=(i,)) for i in range(PARALLEL)]
        [x.start() for x in th]; [x.join() for x in th]
        dt2 = time.time() - t2
        print("[[DSV4]] %d параллельных запросов: токенов %d за %.1f с = %.1f ток/с суммарно"
              % (PARALLEL, sum(res), dt2, sum(res) / max(dt2, 1e-9)), flush=True)
    print("[[DSV4]] хвост лога сервера:", flush=True)
    print(sh("tail -40 /kaggle/working/llama-server.log"), flush=True)
    try:
        proc.terminate()
    except Exception:
        pass
print("[[DSV4]] проба завершена", flush=True)
